In [0]:
-- =============================================================================
-- PHASE 3: GHOST BOOKING DIAGNOSTIC SCRIPT
-- Objective: Prove the 0% conversion rate is due to unaligned synthetic data 
--            in samples.wanderbricks, not a defect in the sessionisation logic.
-- =============================================================================

USE CATALOG meridian_t02;

-- -----------------------------------------------------------------------------
-- STEP 1: The Baseline Check
-- Proves the 30-minute window logic successfully filtered out unaligned records.
-- -----------------------------------------------------------------------------
SELECT 
    COUNT(*) AS total_sessions,
    COUNT(CASE WHEN is_booked = 1 THEN 1 END) AS booked_sessions,
    COUNT(DISTINCT booking_id) AS linked_bookings
FROM gold.fct_sessions;


-- -----------------------------------------------------------------------------
-- STEP 2: Time Gap Analysis
-- Calculates the hours between a user's session and their booking. 
-- Reveals massive disconnects (e.g., bookings occurring months BEFORE any session).
-- -----------------------------------------------------------------------------
WITH BookingUser AS (
    SELECT user_id, created_at AS booking_time 
    FROM silver.silver_bookings 
    LIMIT 1
)
SELECT 
    s.user_id,
    s.session_start_time,
    s.session_end_time,
    b.booking_time,
    TIMESTAMPDIFF(HOUR, s.session_end_time, b.booking_time) AS gap_in_hours
FROM gold.fct_sessions s
JOIN BookingUser b ON s.user_id = b.user_id;


-- -----------------------------------------------------------------------------
-- STEP 3: "Ghost Booking" Proof (Lifetime Clicks)
-- Proves the data generator created bookings without corresponding clicks.
-- e.g., User 9375 booked in April, but their first ever click was in September.
-- -----------------------------------------------------------------------------
SELECT 
    user_id,
    COUNT(*) AS total_lifetime_clicks,
    MIN(event_timestamp) AS first_ever_click,
    MAX(event_timestamp) AS last_ever_click
FROM silver.silver_clickstream
WHERE user_id = 9375
GROUP BY user_id;


-- -----------------------------------------------------------------------------
-- STEP 4: Relaxed Attribution Test
-- Tests if a loose "Same-Day" window fixes the 0% rate. 
-- Result: The mock data is so randomized it fails even same-day attribution.
-- -----------------------------------------------------------------------------
WITH RelaxedAttribution AS (
    SELECT 
        s.session_id,
        s.user_id,
        s.session_start_time,
        b.booking_id
    FROM gold.fct_sessions s
    LEFT JOIN silver.silver_bookings b 
        ON s.user_id = b.user_id
        -- Loosened rule: Same calendar day attribution
        AND TO_DATE(s.session_start_time) = TO_DATE(b.created_at)
)
SELECT 
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(DISTINCT booking_id) AS attributed_bookings,
    ROUND(COUNT(DISTINCT booking_id) / COUNT(DISTINCT session_id) * 100, 2) AS relaxed_conversion_pct
FROM RelaxedAttribution;